# Advanced Reasoning with Agenkit

Unlock powerful reasoning techniques to make your agents smarter and more reliable.

## What You'll Learn

1. **Chain-of-Thought (CoT)** - Step-by-step reasoning
2. **Tree-of-Thought (ToT)** - Explore multiple paths
3. **Self-Consistency** - Voting for reliability
4. **Comparison** - When to use each technique
5. **Best Practices** - Combining techniques effectively

## Prerequisites

- Completed Tutorial 01: Getting Started
- Understanding of agent composition
- (Optional) OpenAI or Anthropic API key

> **Note**: For a more interactive experience with sliders and reactive visualizations, try the [Marimo version](03-advanced-reasoning.py)!

Let's dive into advanced reasoning! 🧠

## Setup

In [ ]:
import agenkit
from agenkit import Agent, Message
from agenkit.techniques.reasoning import ChainOfThought, TreeOfThought, SelfConsistency
import asyncio

print(f"✅ Agenkit version: {agenkit.__version__}")

## Mock LLM for Demonstrations

We'll create a mock LLM to demonstrate concepts without requiring API keys:

In [ ]:
class MockLLM:
    """Mock LLM that demonstrates reasoning techniques."""
    
    def __init__(self, vary_responses=False):
        self.vary_responses = vary_responses
        self.call_count = 0
    
    async def complete(self, prompt: str) -> str:
        """Return mock response."""
        self.call_count += 1
        
        if "15 * 24" in prompt or "15*24" in prompt or "math" in prompt.lower():
            # Math problem responses
            if self.vary_responses and self.call_count % 3 == 0:
                return """Step 1: Use (15 * 20) + (15 * 4)
Step 2: 15 * 20 = 300
Step 3: 15 * 4 = 60  
Step 4: 300 + 60 = 360
Therefore, 15 * 24 = 360"""
            else:
                return """Let me solve this step by step:

1. First, I'll break down 24 into 20 + 4
2. Multiply 15 × 20 = 300
3. Multiply 15 × 4 = 60
4. Add the results: 300 + 60 = 360

Therefore, 15 × 24 = 360"""
        
        elif "alternative" in prompt.lower() or "branches" in prompt.lower():
            # ToT branching responses
            responses = [
                "Approach 1: Break into subproblems and solve iteratively",
                "Approach 2: Identify patterns and apply known solutions",
                "Approach 3: Work backwards from the goal",
            ]
            return responses[self.call_count % len(responses)]
        
        return "Let me think through this carefully..."

print("✅ Created MockLLM")

## 1. Chain-of-Thought (CoT)

### What is Chain-of-Thought?

CoT encourages LLMs to show their reasoning step-by-step, improving accuracy on complex tasks.

**Key Benefits:**
- Improved accuracy on reasoning tasks
- Transparent reasoning process
- Easier to debug and validate
- Works with any LLM

In [ ]:
# Create Chain-of-Thought agent
llm = MockLLM()
cot = ChainOfThought(llm=llm)

# Process a math problem
query = "What is 15 * 24?"
response = await cot.process(Message(role="user", content=query))

print("🔍 Chain-of-Thought Reasoning")
print("=" * 50)
print(f"Query: {query}")
print(f"\nResponse:\n{response.content}")
print(f"\n📊 Metadata:")
print(f"  - Reasoning steps: {response.metadata['num_steps']}")
print(f"  - Technique: {response.metadata['technique']}")

### Extracting Reasoning Steps

CoT automatically parses and tracks reasoning steps:

In [ ]:
print("🔢 Extracted Reasoning Steps:")
print("=" * 50)
for i, step in enumerate(response.metadata['reasoning_steps'], 1):
    print(f"{i}. {step}")

### Custom Prompt Templates

Customize the CoT prompt for your domain:

In [ ]:
# Custom prompt for code debugging
custom_prompt = """Analyze this problem systematically and solve it step by step:

{query}

Show your detailed reasoning:"""

cot_custom = ChainOfThought(llm=MockLLM(), prompt_template=custom_prompt)

response = await cot_custom.process(
    Message(role="user", content="Calculate 15 * 24")
)

print("✨ Custom CoT Prompt:")
print(response.content[:200] + "...")

## 2. Tree-of-Thought (ToT)

### What is Tree-of-Thought?

ToT explores multiple reasoning paths simultaneously, finding the best solution through search.

**Key Benefits:**
- Explores alternatives before committing
- Better for creative/planning tasks
- Can backtrack from dead ends
- Finds higher-quality solutions

**Trade-offs:**
- More expensive (multiple LLM calls)
- Slower than linear reasoning
- Best for non-trivial problems

### Basic Tree-of-Thought Example

In [ ]:
def simple_evaluator(text: str) -> float:
    """Evaluate reasoning quality (0.0-1.0)."""
    score = 0.0
    
    # Length component
    score += min(len(text) / 300, 0.4)
    
    # Structure component
    if any(marker in text for marker in ["1.", "Step", "-"]):
        score += 0.3
    
    # Quality keywords
    keywords = ["approach", "solution", "consider", "therefore"]
    score += sum(0.1 for kw in keywords if kw.lower() in text.lower())
    
    return min(score, 1.0)

# Create Tree-of-Thought agent
tot = TreeOfThought(
    llm=MockLLM(),
    branching_factor=3,  # Explore 3 alternatives per step
    max_depth=2,         # Up to 2 reasoning levels
    evaluator=simple_evaluator,
    strategy="best-first"
)

query = "Design a strategy for optimizing database queries"
response = await tot.process(Message(role="user", content=query))

print("🌳 Tree-of-Thought Reasoning")
print("=" * 50)
print(f"Query: {query}")
print(f"\nBest Path Score: {response.metadata['best_score']:.2f}")
print(f"\nBest Reasoning Path:")
for i, step in enumerate(response.metadata['reasoning_path'], 1):
    print(f"  {i}. {step}")

# Tree statistics
stats = response.metadata['reasoning_tree_stats']
print(f"\n📊 Tree Statistics:")
print(f"  - Total nodes explored: {stats['total_nodes']}")
print(f"  - Max depth reached: {stats['max_depth']}")
print(f"  - Leaf nodes: {stats['num_leaves']}")

### Search Strategies

ToT supports different search strategies:

In [ ]:
strategies = ["best-first", "bfs", "dfs"]

print("🔍 Comparing Search Strategies")
print("=" * 50)

for strategy in strategies:
    tot_strategy = TreeOfThought(
        llm=MockLLM(),
        branching_factor=2,
        max_depth=2,
        strategy=strategy
    )
    
    response = await tot_strategy.process(
        Message(role="user", content="Plan a software architecture")
    )
    
    stats = response.metadata['reasoning_tree_stats']
    print(f"\n{strategy.upper()}:")
    print(f"  - Nodes explored: {stats['total_nodes']}")
    print(f"  - Best score: {response.metadata.get('best_score', 0):.2f}")

print("\n💡 Strategy Guide:")
print("  - best-first: Explores most promising paths first (recommended)")
print("  - bfs: Explores breadth before depth (complete search)")
print("  - dfs: Explores depth before breadth (memory efficient)")

## 3. Self-Consistency

### What is Self-Consistency?

Self-Consistency generates multiple reasoning paths and uses voting to find the most reliable answer.

**Key Benefits:**
- Improved reliability through consensus
- Reduces impact of individual errors
- Confidence scoring included
- Works with any reasoning technique

**Best for:**
- Tasks with objective answers
- Critical decisions requiring confidence
- Reducing hallucinations

In [ ]:
# Wrap CoT with Self-Consistency
cot_base = ChainOfThought(llm=MockLLM(vary_responses=True))
sc = SelfConsistency(
    agent=cot_base,
    num_samples=5,
    voting_strategy="majority"
)

query = "What is 15 * 24?"
response = await sc.process(Message(role="user", content=query))

print("🗳️  Self-Consistency Reasoning")
print("=" * 50)
print(f"Query: {query}")
print(f"\nConsensus Answer: {response.content}")
print(f"\n📊 Consistency Metrics:")
print(f"  - Samples generated: {response.metadata['num_samples']}")
print(f"  - Consistency score: {response.metadata['consistency_score']:.2%}")
print(f"  - Voting strategy: {response.metadata['voting_strategy']}")
print(f"\n📈 Answer Distribution:")
for answer, count in response.metadata['answer_counts'].items():
    print(f"  - '{answer}': {count} votes")

### Voting Strategies

Self-Consistency supports different voting strategies:

In [ ]:
voting_strategies = ["majority", "weighted", "first"]

print("🗳️  Comparing Voting Strategies")
print("=" * 50)

for strategy in voting_strategies:
    sc_strategy = SelfConsistency(
        agent=ChainOfThought(llm=MockLLM(vary_responses=True)),
        num_samples=5,
        voting_strategy=strategy
    )
    
    response = await sc_strategy.process(
        Message(role="user", content="Calculate 15 * 24")
    )
    
    print(f"\n{strategy.upper()}:")
    print(f"  - Consensus: {response.content}")
    print(f"  - Consistency: {response.metadata['consistency_score']:.2%}")

print("\n💡 Strategy Guide:")
print("  - majority: Select most common answer (recommended)")
print("  - weighted: Weight by response quality/length")
print("  - first: No voting, use first sample (baseline)")

## 4. Technique Comparison

Let's compare all three techniques on the same problem:

In [ ]:
import time

query = "What is 15 * 24?"
message = Message(role="user", content=query)

print("📊 Technique Comparison")
print("=" * 60)
print(f"Query: {query}\n")

# 1. Chain-of-Thought
start = time.time()
cot_result = await ChainOfThought(llm=MockLLM()).process(message)
cot_time = time.time() - start

print("🔗 Chain-of-Thought:")
print(f"  - Answer: {cot_result.content.split('Therefore')[-1].strip() if 'Therefore' in cot_result.content else '360'}")
print(f"  - Steps: {cot_result.metadata['num_steps']}")
print(f"  - Time: {cot_time:.3f}s")
print(f"  - Cost: 1 LLM call")
print(f"  - Best for: Fast, straightforward reasoning")

# 2. Tree-of-Thought
start = time.time()
tot_result = await TreeOfThought(
    llm=MockLLM(),
    branching_factor=2,
    max_depth=2
).process(message)
tot_time = time.time() - start

stats = tot_result.metadata['reasoning_tree_stats']
print(f"\n🌳 Tree-of-Thought:")
print(f"  - Best path score: {tot_result.metadata['best_score']:.2f}")
print(f"  - Nodes explored: {stats['total_nodes']}")
print(f"  - Time: {tot_time:.3f}s")
print(f"  - Cost: {stats['total_nodes']} LLM calls")
print(f"  - Best for: Creative tasks, planning, exploration")

# 3. Self-Consistency
start = time.time()
sc_result = await SelfConsistency(
    agent=ChainOfThought(llm=MockLLM(vary_responses=True)),
    num_samples=5
).process(message)
sc_time = time.time() - start

print(f"\n🗳️  Self-Consistency:")
print(f"  - Consensus: {sc_result.content}")
print(f"  - Confidence: {sc_result.metadata['consistency_score']:.2%}")
print(f"  - Samples: {sc_result.metadata['num_samples']}")
print(f"  - Time: {sc_time:.3f}s")
print(f"  - Cost: {sc_result.metadata['num_samples']} LLM calls")
print(f"  - Best for: High-reliability, critical decisions")

print("\n" + "=" * 60)

## 5. When to Use Each Technique

### Decision Matrix

| Scenario | Recommended Technique | Why |
|----------|----------------------|-----|
| **Math problems** | Self-Consistency + CoT | High accuracy, objective answer |
| **Creative writing** | Tree-of-Thought | Explore alternatives |
| **Planning** | Tree-of-Thought | Evaluate multiple strategies |
| **Q&A** | Chain-of-Thought | Fast, transparent |
| **Critical decisions** | Self-Consistency + ToT | Maximum reliability |
| **Code debugging** | Chain-of-Thought | Step-by-step analysis |
| **Design problems** | Tree-of-Thought | Explore design space |
| **Fast responses** | Chain-of-Thought | Single pass |

### Cost vs. Quality Trade-offs

In [ ]:
print("💰 Cost-Quality Trade-offs")
print("=" * 50)
print("\nLow Cost, Fast:")
print("  ✓ Chain-of-Thought")
print("  - 1 LLM call")
print("  - Best for: High volume, real-time")

print("\nMedium Cost, Better Quality:")
print("  ✓ Self-Consistency (3-5 samples)")
print("  - 3-5 LLM calls")
print("  - Best for: Important decisions")

print("\nHigh Cost, Highest Quality:")
print("  ✓ Tree-of-Thought + Self-Consistency")
print("  - Many LLM calls (branching × samples)")
print("  - Best for: Critical, complex problems")

print("\n💡 Recommendation:")
print("  Start with CoT, add Self-Consistency for reliability,")
print("  use ToT for creative/planning tasks.")

## 6. Combining Techniques

You can combine techniques for maximum power:

In [ ]:
# Example: Tree-of-Thought with Self-Consistency
# (Each ToT path uses Self-Consistency for reliability)

print("🔥 Combining Tree-of-Thought + Self-Consistency")
print("=" * 50)

# Base: Chain-of-Thought
base_cot = ChainOfThought(llm=MockLLM(vary_responses=True))

# Layer 1: Add Self-Consistency for reliability
reliable_agent = SelfConsistency(
    agent=base_cot,
    num_samples=3,
    voting_strategy="majority"
)

# Layer 2: Wrap in Tree-of-Thought for exploration
# Note: In practice, you'd use ToT differently, this is conceptual

print("\nArchitecture:")
print("  CoT (base reasoning)")
print("  ↓")
print("  Self-Consistency (voting, 3 samples)")
print("  ↓")
print("  Multiple branches explored")

response = await reliable_agent.process(
    Message(role="user", content="Calculate 15 * 24")
)

print(f"\nResult: {response.content}")
print(f"Confidence: {response.metadata['consistency_score']:.2%}")
print("\n💡 Use case: Mission-critical decisions with high accuracy requirements")

## 7. Best Practices

### 1. Start Simple

In [ ]:
print("✅ Best Practices for Advanced Reasoning")
print("=" * 50)

print("\n1. Progressive Enhancement:")
print("   - Start with CoT for baseline")
print("   - Add Self-Consistency if accuracy is low")
print("   - Try ToT if problem is creative/exploratory")

print("\n2. Monitor Costs:")
print("   - Track LLM call counts in metadata")
print("   - Set max_depth and branching_factor appropriately")
print("   - Use num_samples wisely (3-7 usually sufficient)")

print("\n3. Evaluate Quality:")
print("   - Check consistency_score (aim for >0.7)")
print("   - Review reasoning_steps for CoT")
print("   - Examine answer_counts for agreement")

print("\n4. Domain-Specific Tuning:")
print("   - Customize prompt_template for your domain")
print("   - Create custom evaluator functions")
print("   - Adjust answer_extractor for your format")

print("\n5. Testing:")
print("   - Benchmark on representative problems")
print("   - Compare techniques on your use case")
print("   - Measure accuracy vs. cost trade-offs")

## Summary

You've mastered advanced reasoning techniques! 🎉

✅ **Chain-of-Thought** - Fast, transparent step-by-step reasoning  
✅ **Tree-of-Thought** - Explore alternatives, creative solutions  
✅ **Self-Consistency** - Voting for reliability and confidence  
✅ **Comparison** - Know when to use each technique  
✅ **Combining** - Stack techniques for maximum power  

## Next Steps

- **[Marimo Version](03-advanced-reasoning.py)** - Interactive notebook with sliders for parameters
- **[Deployment Guide](../docs/deployment.md)** - Deploy reasoning agents to production
- **[Evaluation Framework](../docs/evaluation.md)** - Measure reasoning quality
- **[Advanced Examples](https://github.com/scttfrdmn/agenkit/tree/main/examples/techniques/reasoning)** - More reasoning patterns

## Quick Reference

```python
# Chain-of-Thought
cot = ChainOfThought(llm=my_llm)
response = await cot.process(message)

# Tree-of-Thought
tot = TreeOfThought(
    llm=my_llm,
    branching_factor=3,
    max_depth=3,
    strategy="best-first"
)
response = await tot.process(message)

# Self-Consistency
sc = SelfConsistency(
    agent=cot,
    num_samples=5,
    voting_strategy="majority"
)
response = await sc.process(message)
```

Ready to build intelligent reasoning systems! 🧠🚀